In [ ]:

from snowflake.snowpark.context import get_active_session

session = get_active_session()

print("Snowflake connection successful!")
print("Database:", session.get_current_database())
print("Schema:", session.get_current_schema())
print("Warehouse:", session.get_current_warehouse())

# Check existing tables in the project schema
session.sql("SHOW TABLES IN SCHEMA INSURANCE_ML.ML_PIPE").show()


In [ ]:
session.table("INSURANCE_DATA").count()

In [ ]:
session.table("INSURANCE_DATA").print_schema()

In [ ]:
import pandas as pd
import numpy as np

df = session.table("INSURANCE_DATA").to_pandas()

df.columns = df.columns.str.lower()

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

df.head()

In [ ]:
from sklearn.model_selection import train_test_split

# Remove duplicate rows from the working DataFrame
df = df.drop_duplicates().copy()

# Separate features and target
X = df.drop(columns=["charges"])
y = df["charges"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Clean dataset:", df.shape)
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# Identify columns
numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

# Build the complete pipeline
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            random_state=42,
            objective="reg:squarederror"
        ))
    ]
)

# Train the model
model_pipeline.fit(X_train, y_train)

print("Model trained successfully!")

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

# Predict on unseen test data
predictions = model_pipeline.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 4))

In [ ]:


from snowflake.ml.registry import Registry
from snowflake.ml.model import target_platform

registry = Registry(
    session=session,
    database_name="INSURANCE_ML",
    schema_name="ML_PIPE"
)

model_ref = registry.log_model(
    model=model_pipeline,
    model_name="INSURANCE_CHARGES_MODEL",
    comment="XGBoost insurance charges prediction model",
    pip_requirements=["scikit-learn", "xgboost"],
    target_platforms=[target_platform.TargetPlatform.WAREHOUSE],
    sample_input_data=X_train.head(5)
)

print("Model registered successfully!")
print(model_ref)


In [ ]:
models = registry.get_model("INSURANCE_CHARGES_MODEL")
models.show_versions()

In [ ]:
model_v1 = registry.get_model(
    "INSURANCE_CHARGES_MODEL"
).version("PRETTY_WORM_2")

registered_predictions = model_v1.run(
    X_test,
    function_name="predict"
)

print(registered_predictions.head())

In [ ]:

# Step 9: Create input and output tables

session.sql("""
CREATE TABLE IF NOT EXISTS INSURANCE_INPUT (
    AGE INT,
    SEX STRING,
    BMI FLOAT,
    CHILDREN INT,
    SMOKER STRING,
    REGION STRING
)
""").collect()

session.sql("""
CREATE TABLE IF NOT EXISTS INSURANCE_OUTPUT (
    AGE INT,
    SEX STRING,
    BMI FLOAT,
    CHILDREN INT,
    SMOKER STRING,
    REGION STRING,
    PREDICTED_CHARGES FLOAT
)
""").collect()

print("Input and output tables created successfully!")


In [ ]:

# Step 10: Create a stream on the input table

session.sql("""
CREATE STREAM IF NOT EXISTS INSURANCE_INPUT_STREAM
ON TABLE INSURANCE_INPUT
""").collect()

print("Stream created successfully!")


In [ ]:

# Fix: align input column names with the model signature

procedure_sql = """
CREATE OR REPLACE PROCEDURE PROCESS_INSURANCE_DATA()
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('snowflake-snowpark-python', 'snowflake-ml-python', 'pandas')
HANDLER = 'process_insurance'
EXECUTE AS OWNER
AS
$$
def process_insurance(session):
    from snowflake.ml.registry import Registry

    input_df = session.table("INSURANCE_INPUT_STREAM").to_pandas()

    if input_df.empty:
        return "No new records to process."

    features = input_df[
        ["AGE", "SEX", "BMI", "CHILDREN", "SMOKER", "REGION"]
    ].copy()

    # The registered model expects lowercase feature names
    features.columns = [col.lower() for col in features.columns]

    registry = Registry(
        session=session,
        database_name="INSURANCE_ML",
        schema_name="ML_PIPE"
    )

    model = registry.get_model(
        "INSURANCE_CHARGES_MODEL"
    ).version("PRETTY_WORM_2")

    predictions = model.run(
        features,
        function_name="predict"
    )

    # Restore uppercase names for the output table
    result = features.copy()
    result.columns = [col.upper() for col in result.columns]
    result["PREDICTED_CHARGES"] = predictions.iloc[:, 0].values

    session.create_dataframe(result).write.save_as_table(
        "INSURANCE_OUTPUT",
        mode="append"
    )

    return f"Processed {len(result)} records successfully."
$$
"""

session.sql(procedure_sql).collect()
print("Stored procedure updated.")


In [ ]:
result = session.sql(
    "CALL PROCESS_INSURANCE_DATA()"
).collect()

print(result)

In [ ]:
session.table("INSURANCE_OUTPUT").show()

In [ ]:

# Step 13: Create and activate the automated task

session.sql("""
CREATE OR REPLACE TASK INSURANCE_AUTO_TASK
WAREHOUSE = ML_PIPELINE_WH
WHEN SYSTEM$STREAM_HAS_DATA('INSURANCE_INPUT_STREAM')
AS
CALL PROCESS_INSURANCE_DATA()
""").collect()

session.sql("""
ALTER TASK INSURANCE_AUTO_TASK RESUME
""").collect()

print("Automated task created and resumed!")


In [ ]:
session.sql("SHOW TASKS LIKE 'INSURANCE_AUTO_TASK'").show()

In [ ]:
session.sql("""
INSERT INTO INSURANCE_INPUT
(AGE, SEX, BMI, CHILDREN, SMOKER, REGION)
VALUES
(40, 'male', 30.5, 2, 'no', 'southeast')
""").collect()

print("New test record inserted.")

In [ ]:
session.table("INSURANCE_OUTPUT").show()

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(
    session=session,
    database_name="INSURANCE_ML",
    schema_name="ML_PIPE"
)

model = registry.get_model("INSURANCE_CHARGES_MODEL")
model.show_versions()

In [ ]:
model_v2 = registry.get_model(
    "INSURANCE_CHARGES_MODEL"
).version("PRETTY_WORM_2")

print(model_v2)